In [ ]:
import pandas as pd
import scanpy as sc

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.obs["transcript_UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["transcript_UMAP2"] = adata.obsm["X_umap"][:, 1]

In [ ]:
te_clustering = pd.read_csv("te_leiden.csv", index_col=0).rename(columns={"leiden": "leiden_te"})
te_clustering["leiden_te"] = te_clustering["leiden_te"].astype(int)
te_clustering.loc[lambda x: x["leiden_te"] >= 20, "leiden_te"] = 20
te_clustering["leiden_te"] = te_clustering["leiden_te"].astype(str)
te_clustering.value_counts()

In [ ]:
adata.obs = adata.obs.join(te_clustering)

In [ ]:
sc.pl.umap(adata, color="leiden_te", size=20)

In [ ]:
for leiden_te in adata.obs["leiden_te"].unique():
    adata_subb = adata[adata.obs["leiden_te"] == leiden_te].copy()
    sc.pl.umap(adata_sub, color="leiden_te", size=20)

In [ ]:
adata_sub1 = adata[adata.obs["leiden_te"] != "20"].copy()
adata_sub1

In [ ]:
adata_sub2 = adata[adata.obs["leiden_te"] == "20"].copy()

In [ ]:
sc.pl.heatmap(
    adata_sub2,
    var_names=adata.var.index[adata.var.index.str.startswith("in")],
    groupby="leiden_te",
)

In [ ]:
sc.pl.heatmap(
    adata_sub1,
    var_names=adata.var.index[adata.var.index.str.startswith("in")],
    groupby="leiden_te",
)

In [ ]:
sc.pp.neighbors(adata_sub1, use_rep="X_scVI")
sc.tl.umap(adata_sub1)

In [ ]:
sc.pl.umap(adata_sub1)

In [ ]:
for gene in adata.var[adata.var.index.str.startswith("in")].index:
    print(gene)

## Clustering

In [ ]:
adata.X = adata.layers["reads"].copy()
# adata_sub = adata[:, adata.var.index.str.startswith("in")].copy()
adata_sub = adata[:, adata.var.index.str.contains(r"^(ins|int|tnp|xis|rhs)", regex=True)]
# sc.pp.normalize_total(adata_sub)
sc.pp.log1p(adata_sub)
sc.pp.neighbors(adata_sub, use_rep="X")
sc.tl.umap(adata_sub)
sc.pl.umap(adata_sub)

In [ ]:
sc.tl.leiden(adata_sub)
sc.pl.umap(adata_sub, color="leiden")
adata_sub.obs["picky_cluster"] = adata_sub.obs["leiden"] == "1"
sc.pl.umap(adata_sub, color="picky_cluster")
adata_sub.obs["leiden"].to_csv("te_leiden.csv")